# 阶段 4 · B：从我们的硬标签 1.5B 出发做 GRPO（Colab）

起点是阶段 3 的 **hard_1.5b**（压缩格式，答案是一个字母，只用最优动作当标签训练）。用 fixed 奖励做 GRPO，看它能不能追上软标签的 soft_1.5b。预期打平（推断，没测过）：两个学生都已经在公式上限附近。

和 A 的区别：这里每个回答只有 **1 个 token**（一个字母），所以同一道题的 G 个回答共用一次前向：前向一次拿到最后一个位置的分布，从里面抽 G 个字母，损失也在这一个位置上算。不用 `generate`，比 A 快得多。

只有一次运行，不用改任何格，直接「代码执行程序 → 全部运行」。G4 上预计 10–20 分钟（估算）。依据：阶段 3 hard_1.5b 在 G4 上整个运行实测约 14 分钟，其中训练 4 万条、评测 2 × 7658 条、导出 GGUF；这里训练 1000 步 × 8 道题，每道题两次前向（当前模型和参考模型），评测一次约 7700 条。

跑之前：运行时选 **G4**（没有就 L4 或 A100）。Drive 里要有：
- `MyDrive/COMLLM/s3_data/`（阶段 3 的数据，不用动）
- `MyDrive/COMLLM/s3_runs/hard_1.5b_<时间>/adapter/`（阶段 3 训好的 LoRA，第 3 格自动找最新的那个）
- `MyDrive/COMLLM/s4_data/`（A 的步骤 2 上传的；这里只用其中的 `grpo_reward.py`）

**结果**在 Drive 的 `MyDrive/COMLLM/s4_runs/hard_grpo_<UTC 时间>/`。

**止损线**：
- 第 7 格开头先在验证集上评一次起点。准确率低于 90% 会自动报错停下：说明 LoRA 没读对或者提示词拼错了（没训练的 1.5B 约 60%）。
- 训练第 20 步会打印预计时长。超过 45 分钟就停下来告诉我。
- 任何一格报错就停，截图发到对话里。
- 跑完「代码执行程序 → 断开连接并删除运行时」。

In [ ]:
# 第 1 格：安装固定版本的库（和阶段 3 一样）
import subprocess, sys
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "transformers==5.20.2", "peft==0.21.2", "accelerate", "sentencepiece"], check=True)
    # Colab 预装的 torchao 版本太老，peft 0.21 检测到它会直接报错。这里用不到 torchao，卸掉
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], check=False)

import importlib, importlib.util
importlib.invalidate_caches()
assert importlib.util.find_spec("torchao") is None, "torchao 还在：菜单「代码执行程序 → 重新启动会话」后再「全部运行」"
print("依赖安装完成")

In [ ]:
# 第 2 格：检查 GPU。G4 / L4 / A100 支持 bf16；T4 不支持，这个 notebook 不在 T4 上跑
import os, time, json, math, hashlib, csv, glob
from datetime import datetime, timezone
import numpy as np
import torch

if IN_COLAB:
    subprocess.run(["nvidia-smi"], check=False)
    assert torch.cuda.is_available(), "没检测到 GPU：菜单「代码执行程序 → 更改运行时类型」选 G4 后再运行"
HAS_GPU = torch.cuda.is_available()
DEVICE = "cuda" if HAS_GPU else "cpu"
if HAS_GPU:
    assert torch.cuda.is_bf16_supported(), "这块 GPU 不支持 bf16（T4 就不支持）：换成 G4 或 L4"
DTYPE = torch.bfloat16 if HAS_GPU else torch.float32
GPU_NAME = torch.cuda.get_device_name() if HAS_GPU else "无（CPU）"
print("GPU:", GPU_NAME, "| 精度:", "bf16" if HAS_GPU else "fp32", "| torch", torch.__version__)

In [ ]:
# 第 3 格：设置（不用改）
RUN = "hard_grpo"
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"                     # 和阶段 3 同一个基座
REVISION = "989aa7980e4cf806f80c7fef2b1adb7bc71aa306"       # 钉死的提交号，见 docs/versions.md

# GRPO 超参数：和 A 完全一样
G = 8                      # 每道题采样几个回答（论文式 17 的 G）
PROMPTS_PER_STEP = 8       # 每步几道题：一步一共 8 × 8 = 64 个回答
TEMP = 1.3                 # 采样温度
TOP_P = 0.95               # 采样时只在累计概率前 95% 的 token 里抽
BETA = 0.04                # KL 惩罚的系数 β（论文式 19）
LR = 5e-5                  # 学习率（前 WARMUP_STEPS 步线性升上去，之后不变）
WARMUP_STEPS = 10
EPS_CLIP = 0.2             # 论文式 19 的 clip 范围 ε
LORA_R, LORA_ALPHA = 16, 32
# B 自己的设置
STEPS = 1000               # 训练步数：1000 × 8 = 8000 道题，从阶段 3 训练集里随机取，不重复
EVAL_EVERY = 100           # 每多少步在验证集上评一次
N_VAL = 400                # 验证集取前 400 条（和阶段 3 训练中途看的一样）
EVAL_BATCH = 32            # 评测时每批几条
MIN_START_ACC = 90.0       # 起点在验证集上的准确率低于这个就停（%）
SEED = 0
DATA_SEED = 4              # 选训练题用的随机种子

# 本机测试用（Colab 上这些环境变量都不存在，不起作用）
LOCAL_MODEL = os.environ.get("S4_LOCAL_MODEL")
for name in ("STEPS", "EVAL_EVERY", "N_VAL", "PROMPTS_PER_STEP", "G"):
    if os.environ.get("S4_" + name):
        globals()[name] = int(os.environ["S4_" + name])
if os.environ.get("S4_MIN_START_ACC"):
    MIN_START_ACC = float(os.environ["S4_MIN_START_ACC"])

STAMP = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA3 = "/content/drive/MyDrive/COMLLM/s3_data"
    DATA4 = "/content/drive/MyDrive/COMLLM/s4_data"
    S3_RUNS = "/content/drive/MyDrive/COMLLM/s3_runs"
    OUT = f"/content/drive/MyDrive/COMLLM/s4_runs/{RUN}_{STAMP}"
else:
    DATA3 = os.environ["S4_DATA3"]
    DATA4 = os.environ["S4_DATA4"]
    S3_RUNS = os.environ["S4_S3_RUNS"]
    OUT = os.path.join(os.environ["S4_OUT"], f"{RUN}_{STAMP}")
os.makedirs(OUT, exist_ok=True)

# 起点：s3_runs 里最新的 hard_1.5b_<时间>/（文件夹名里的时间越新，排序越靠后）
found = []
for d in sorted(glob.glob(os.path.join(S3_RUNS, "hard_1.5b_*"))):
    if os.path.exists(os.path.join(d, "adapter", "adapter_config.json")):
        found.append(d)
assert found, f"在 {S3_RUNS} 里找不到 hard_1.5b_<时间>/adapter/：确认阶段 3 的 hard_1.5b 结果还在 Drive 上"
START_DIR = found[-1]
ADAPTER = os.path.join(START_DIR, "adapter")
print("起点（阶段 3 的硬标签 1.5B）:", START_DIR)
print("结果目录:", OUT)

In [ ]:
# 第 4 格：读数据，核对校验值；读奖励函数（和 Mac 仓库里 train/grpo_reward.py 是同一个文件）
def read_jsonl(path):
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    return rows


def md5_of(path):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()


with open(os.path.join(DATA3, "meta.json"), encoding="utf-8") as f:
    META3 = json.load(f)
with open(os.path.join(DATA4, "meta.json"), encoding="utf-8") as f:
    META4 = json.load(f)
DATA = {}
for name in ("train", "val", "test_seen", "test_unseen", "friend_clean"):
    path = os.path.join(DATA3, name + ".jsonl")
    assert md5_of(path) == META3["files"][name]["md5"], f"{name}.jsonl 的校验值和 meta.json 对不上：重新上传 s3_data"
    DATA[name] = read_jsonl(path)
    for i, r in enumerate(DATA[name]):
        r["_idx"] = i                      # 在文件里是第几行，preds.csv 用它对应
        r["split"] = name
    print(f"{name:13s} {len(DATA[name]):6d} 条 ✓")

assert md5_of(os.path.join(DATA4, "grpo_reward.py")) == META4["reward_md5"], "grpo_reward.py 和 meta.json 对不上：重新上传 s4_data"
sys.path.insert(0, DATA4)
import grpo_reward as R
importlib.reload(R)

SYSTEM = META3["compact_system"]
LETTERS = META3["letters"]                 # A = 本地，B = 服务器 1，……
DEADLINE = float(META3["deadline_slots"])  # 截止期 10 槽
assert LETTERS == R.LETTERS


def chatml(messages):
    """和仓库里 deploy/prompts.py 的 chatml 一模一样：Jetson 上就是这样拼提示词的。"""
    text = ""
    for m in messages:
        text = text + "<|im_start|>" + m["role"] + "\n" + m["content"] + "<|im_end|>\n"
    text = text + "<|im_start|>assistant\n"
    return text


def compact_messages(row):
    return [{"role": "system", "content": SYSTEM}, {"role": "user", "content": row["user"]}]


# 训练题：训练集打乱后取前 STEPS × PROMPTS_PER_STEP 条
rng_data = np.random.default_rng(DATA_SEED)
order = rng_data.permutation(len(DATA["train"]))
TRAIN_ROWS = []
for i in order[:STEPS * PROMPTS_PER_STEP]:
    TRAIN_ROWS.append(DATA["train"][int(i)])
VAL_ROWS = DATA["val"][:N_VAL]
EVAL_SPLITS = ("val", "test_seen", "test_unseen", "friend_clean")
for split in EVAL_SPLITS:
    for r in DATA[split]:
        r["text"] = chatml(compact_messages(r))
for r in TRAIN_ROWS:
    r["text"] = chatml(compact_messages(r))
count = {}
for r in TRAIN_ROWS:
    count[r["servers"]] = count.get(r["servers"], 0) + 1
print(f"训练题 {len(TRAIN_ROWS)} 条，按台数：{dict(sorted(count.items()))}；验证集 {len(VAL_ROWS)} 条")

In [ ]:
# 第 5 格：分词器、起点模型、按部署规则作答（这一格只定义函数，不跑东西）
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel


def load_tokenizer():
    """读分词器，做和阶段 3 一样的三项核对（任何一项不过都会直接报错停下）。

    1. 12 个动作字母各是 1 个 token
    2. Hugging Face 的对话模板拼出来的文字 = chatml 拼的 = Mac 上用仓库代码拼的（meta.json 里的样例）
    3. 答案字母紧跟在提示词后面，单独成 1 个 token
    返回 (分词器, 12 个字母的 token 编号)。
    """
    if LOCAL_MODEL:
        tk = AutoTokenizer.from_pretrained(LOCAL_MODEL)
    else:
        tk = AutoTokenizer.from_pretrained(MODEL_ID, revision=REVISION)
    letter_ids = []
    for c in LETTERS:
        ids = tk.encode(c, add_special_tokens=False)
        assert len(ids) == 1, f"动作 {c} 不是单 token：{ids}"
        letter_ids.append(ids[0])
    for i, golden in enumerate(META3["golden"]["compact"]):
        r = DATA["train"][i]
        assert chatml(compact_messages(r)) == golden, "这里拼的提示词和 Mac 上仓库代码拼的不一样"
        hf = tk.apply_chat_template(compact_messages(r), tokenize=False, add_generation_prompt=True)
        assert hf == golden, "Hugging Face 对话模板拼的提示词和 Mac 上的不一样"
    for r in DATA["train"][:50]:
        text = chatml(compact_messages(r))
        p_ids = tk(text, add_special_tokens=False)["input_ids"]
        a = int(np.argmin(r["costs_compact"]))
        f_ids = tk(text + LETTERS[a], add_special_tokens=False)["input_ids"]
        assert f_ids == p_ids + [letter_ids[a]], "答案字母和提示词末尾粘在了一起"
    return tk, letter_ids


def load_start():
    """起点模型 = 基座 + 阶段 3 的 hard_1.5b LoRA，合并成一个普通模型。

    合并后它就是 GRPO 的参考模型 π_ref：后面再套一层新的 LoRA 训练，关掉这层新 LoRA 就回到它。
    """
    t0 = time.time()
    if LOCAL_MODEL:
        base = AutoModelForCausalLM.from_pretrained(LOCAL_MODEL, dtype=DTYPE)
    else:
        base = AutoModelForCausalLM.from_pretrained(MODEL_ID, revision=REVISION, dtype=DTYPE)
    md = PeftModel.from_pretrained(base, ADAPTER)
    md = md.merge_and_unload()
    md = md.to(DEVICE)
    md.eval()
    print(f"读入起点并合并 LoRA：{time.time() - t0:.0f} 秒")
    return md


def encode_rows(rows):
    """把每行的提示词全文切成 token 编号，存进 r["ids"]（已经有的跳过）。"""
    todo = []
    for r in rows:
        if "ids" not in r:
            todo.append(r)
    texts = []
    for r in todo:
        texts.append(r["text"])
    if texts:
        ids = TOK(texts, add_special_tokens=False)["input_ids"]
        for r, x in zip(todo, ids):
            r["ids"] = x


def batches_by_count(rows, size, rng=None):
    """按服务器台数分组（同组提示词长度几乎一样），每组切成 size 条一批；给了 rng 就打乱。"""
    groups = {}
    for r in rows:
        groups.setdefault(r["servers"], []).append(r)
    out = []
    for k in sorted(groups):
        g = groups[k]
        if rng is not None:
            order = rng.permutation(len(g))
            g = [g[int(i)] for i in order]
        for s in range(0, len(g), size):
            out.append(g[s:s + size])
    if rng is not None:
        order = rng.permutation(len(out))
        out = [out[int(i)] for i in order]
    return out


def collate(batch):
    """左补齐：所有行的最后一个位置都是"assistant\\n"之后、该出答案的地方。

    position_ids 让每行真实的第一个 token 从 0 开始编号，和 Jetson 上一条一条推理时一样。
    """
    width = 0
    for r in batch:
        if len(r["ids"]) > width:
            width = len(r["ids"])
    ids = torch.full((len(batch), width), TOK.pad_token_id, dtype=torch.long)
    mask = torch.zeros((len(batch), width), dtype=torch.long)
    for i, r in enumerate(batch):
        n = len(r["ids"])
        ids[i, width - n:] = torch.tensor(r["ids"], dtype=torch.long)
        mask[i, width - n:] = 1
    pos = (mask.cumsum(dim=1) - 1).clamp(min=0)
    return ids.to(DEVICE), mask.to(DEVICE), pos.to(DEVICE)


def last_logits(md, batch):
    """前向一次，只取最后一个位置的 logits（整个词表的分数），形状 (条数, 词表)。"""
    ids, mask, pos = collate(batch)
    with torch.autocast(device_type="cuda", dtype=torch.bfloat16, enabled=HAS_GPU):
        out = md(input_ids=ids, attention_mask=mask, position_ids=pos, logits_to_keep=1)
    return out.logits[:, -1, :].float()


@torch.no_grad()
def predict(md, rows):
    """按部署规则选动作，返回 {(split, 行号): (动作, 领先幅度, 合法字母总概率, 各合法字母概率)}。

    部署规则（和 deploy/prompts.py、阶段 3 一样）：在整个词表概率最高的 20 个 token 里，
    找概率最大的合法字母（A 到第 n_actions 个字母）；20 个里一个合法字母都没有就算非法，动作记 −1。
    """
    md.eval()
    encode_rows(rows)
    out = {}
    for batch in batches_by_count(rows, EVAL_BATCH):
        probs = torch.softmax(last_logits(md, batch), dim=-1)
        p20 = torch.topk(probs, 20, dim=-1).values[:, -1].cpu().numpy()     # 每行第 20 名的概率
        letter_p = probs[:, LETTER_IDS].cpu().numpy()
        for i, r in enumerate(batch):
            n = r["n_actions"]
            lp = letter_p[i, :n]
            best = int(np.argmax(lp))
            second = 0.0
            for a in range(n):
                if a != best and lp[a] > second:
                    second = float(lp[a])
            margin = math.log(max(float(lp[best]), 1e-30)) - math.log(max(second, 1e-30))
            pred = best if lp[best] >= p20[i] else -1
            out[(r["split"], r["_idx"])] = (pred, margin, float(lp.sum()), [round(float(x), 5) for x in lp])
    return out


def quick_metrics(rows, preds):
    """准确率、PR（论文式 28–29）、非法率、选服务器 1 和本地的比例、模型给最优动作的平均概率。
    正式指标在 Mac 上用 eval.open_loop 算。"""
    hit = 0
    invalid = 0
    s1 = 0
    local = 0
    al = 0.0
    al_best = 0.0
    p_best = 0.0
    for r in rows:
        a, _, _, lp = preds[(r["split"], r["_idx"])]
        p_best += lp[r["best"]]
        if a < 0:
            invalid += 1
            al += DEADLINE + R.RHO
        else:
            al += R.effective_cost(r["costs"][a], DEADLINE)
            if a == r["best"]:
                hit += 1
            if a == 1:
                s1 += 1
            if a == 0:
                local += 1
        al_best += R.effective_cost(r["costs"][r["best"]], DEADLINE)
    n = len(rows)
    return {"n": n, "acc_pct": round(100.0 * hit / n, 1), "pr_pct": round(100.0 * al_best / al, 2),
            "invalid_pct": round(100.0 * invalid / n, 1), "s1_pct": round(100.0 * s1 / n, 1),
            "local_pct": round(100.0 * local / n, 1), "p_best_pct": round(100.0 * p_best / n, 1)}


def write_preds(path, rows, preds):
    """和阶段 3 一样的格式：split, idx, pred（−1 = 非法）, margin, letter_mass。Mac 上的 eval.open_loop 读这个文件。"""
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["split", "idx", "pred", "margin", "letter_mass"])
        for r in rows:
            pred, margin, mass, _ = preds[(r["split"], r["_idx"])]
            w.writerow([r["split"], r["_idx"], pred, round(margin, 4), round(mass, 4)])


def trainable(md):
    """模型里要训练的参数（只有新套上的 LoRA 小矩阵）。"""
    out = []
    for p in md.parameters():
        if p.requires_grad:
            out.append(p)
    return out

In [ ]:
# 第 6 格：GRPO 的一步（这一格只定义函数，不跑东西）
from peft import LoraConfig, get_peft_model


def sample_letters(logits):
    """从最后一个位置的分布里，每道题抽 G 个 token（和 generate 的采样规则一样：先除以温度，再做 top-p）。

    1. p = softmax(logits / TEMP)
    2. 从大到小排序，累计概率；只留下"排在它前面的累计概率还不到 TOP_P"的那些 token，其余概率清零
    3. 剩下的重新归一化，有放回地抽 G 次
    返回 token 编号，形状 (题数, G)。
    """
    p = torch.softmax(logits / TEMP, dim=-1)
    sp, si = torch.sort(p, dim=-1, descending=True)
    before = torch.cumsum(sp, dim=-1) - sp
    sp = torch.where(before < TOP_P, sp, torch.zeros_like(sp))
    keep = torch.zeros_like(p).scatter(-1, si, sp)
    keep = keep / keep.sum(dim=-1, keepdim=True)
    return torch.multinomial(keep, G, replacement=True)


def grpo_step(md, opt, batch, step):
    """一步 GRPO，返回这一步的统计。

    1. 参考模型（关掉新 LoRA = hard_1.5b）前向一次，取最后一个位置的 logits，不需要梯度
    2. 当前模型前向一次（要梯度），取最后一个位置的 logits
    3. 每道题从当前模型的分布里抽 G 个 token（每个回答就是这一个 token）
    4. 每个 token 解析成动作：只有正好是 12 个字母 token 之一才算认出（和部署规则一样，" A" 这种带空格的不算）；
       再算 fixed 奖励（用 costs_compact：模型从提示词里看得到的精度，和阶段 3 的标签一样）
    5. 组内归一化得到优势 A（论文式 18）；奖励全相同的组优势全是 0
    6. 论文式 19 的损失，和 A 的写法一样：
         ρ = π_θ / π_θold = 1（每批只更新一次），clip 不起作用，梯度 = −A × ∇log π_θ(抽到的 token)
         KL 用 exp(ref − lp) − (ref − lp) − 1，在抽到的 token 上估计；对全部 G × 题数 个回答取平均
    7. 梯度裁剪到 1.0 → 设学习率 → AdamW 更新
    """
    # 1. 参考模型
    md.eval()
    with torch.no_grad():
        with md.disable_adapter():
            ref_logits = last_logits(md, batch)
    # 2. 当前模型
    md.train()
    logits = last_logits(md, batch)
    # 3. 采样
    with torch.no_grad():
        tok_ids = sample_letters(logits.detach())
    # 4. 奖励和动作
    rewards = []
    kinds = []
    actions = []
    for q, r in enumerate(batch):
        for j in range(G):
            t = int(tok_ids[q, j])
            text = ""
            if t in LETTER_IDS:
                text = LETTERS[LETTER_IDS.index(t)]
            kind, a = R.parse_letter(text, r["n_actions"])
            rewards.append(R.reward_fixed_action(kind, a, r["costs_compact"], DEADLINE))
            kinds.append(kind)
            actions.append(a)
    # 5. 优势：顺序是 题 1 的 G 个、题 2 的 G 个……，和 tok_ids 按行展开的顺序一样
    adv, zero_std = R.group_advantages(rewards, G)
    adv_t = torch.tensor(adv, dtype=torch.float32, device=DEVICE).view(len(batch), G)
    # 6. 损失
    lp = torch.log_softmax(logits / TEMP, dim=-1).gather(-1, tok_ids)              # (题数, G)
    ref = torch.log_softmax(ref_logits / TEMP, dim=-1).gather(-1, tok_ids)
    ratio = torch.exp(lp - lp.detach())
    per = -torch.minimum(ratio * adv_t, torch.clamp(ratio, 1.0 - EPS_CLIP, 1.0 + EPS_CLIP) * adv_t)
    kl = torch.exp(ref - lp) - (ref - lp) - 1.0
    per = per + BETA * kl
    loss = per.mean()
    loss.backward()
    # 7. 更新
    torch.nn.utils.clip_grad_norm_(trainable(md), 1.0)
    lr = LR * min(1.0, (step + 1) / WARMUP_STEPS)
    for g in opt.param_groups:
        g["lr"] = lr
    opt.step()
    opt.zero_grad(set_to_none=True)

    # 统计（字段名和 A 一样，Mac 上的 eval.grpo_report 两边一起画图）
    n_all = len(rewards)
    n_bad = 0
    n_none = 0
    n_s1 = 0
    n_local = 0
    hit = 0
    for i in range(n_all):
        if kinds[i] == R.BAD_SERVER:
            n_bad += 1
        elif kinds[i] == R.NONE:
            n_none += 1
        if actions[i] == 1:
            n_s1 += 1
        if actions[i] == 0:
            n_local += 1
        if actions[i] == batch[i // G]["best"]:
            hit += 1
    top_share = 0.0
    for g0 in range(0, n_all, G):
        cnt = {}
        for i in range(g0, g0 + G):
            cnt[actions[i]] = cnt.get(actions[i], 0) + 1
        top = 0
        for v in cnt.values():
            if v > top:
                top = v
        top_share += top / G
    with torch.no_grad():
        p1 = torch.softmax(logits.detach(), dim=-1)
        p_best = 0.0
        for q, r in enumerate(batch):
            p_best += float(p1[q, LETTER_IDS[r["best"]]])
    return {"step": step + 1, "loss": round(float(loss.item()), 5), "reward_mean": round(float(np.mean(rewards)), 4),
            "sample_acc_pct": round(100.0 * hit / n_all, 1),
            "zero_std_frac": round(zero_std / len(batch), 3), "kl": round(float(kl.detach().mean().item()), 5),
            "bad_server_pct": round(100.0 * n_bad / n_all, 1), "unparsed_pct": round(100.0 * n_none / n_all, 1),
            "s1_pct": round(100.0 * n_s1 / n_all, 1), "local_pct": round(100.0 * n_local / n_all, 1),
            "group_top_share": round(top_share / len(batch), 3), "p_best_pct": round(100.0 * p_best / len(batch), 1),
            "lr": lr}

In [ ]:
# 第 7 格：开始跑
torch.manual_seed(SEED)
np.random.seed(SEED)
TOK, LETTER_IDS = load_tokenizer()
print("三项核对都通过 ✓  动作字母的 token 编号:", dict(zip(LETTERS, LETTER_IDS)))
SUMMARY = {"run": RUN, "gpu": GPU_NAME, "torch": torch.__version__, "start": START_DIR,
           "settings": {"G": G, "prompts_per_step": PROMPTS_PER_STEP, "temp": TEMP, "top_p": TOP_P, "beta": BETA,
                        "lr": LR, "warmup": WARMUP_STEPS, "lora_r": LORA_R, "lora_alpha": LORA_ALPHA,
                        "steps": STEPS, "reward": "fixed（costs_compact）", "data_seed": DATA_SEED}}

# 1. 起点在验证集上评一次（第 0 步），不对就停
md = load_start()
VAL_LOG = []


def run_val(step):
    v = quick_metrics(VAL_ROWS, predict(md, VAL_ROWS))
    v["step"] = step
    VAL_LOG.append(v)
    print(f"  验证集（部署规则，{v['n']} 条）第 {step} 步：准确率 {v['acc_pct']}%  PR {v['pr_pct']}%  "
          f"非法 {v['invalid_pct']}%  最优动作平均概率 {v['p_best_pct']}%")
    return v


v0 = run_val(0)
assert v0["acc_pct"] >= MIN_START_ACC, (f"起点准确率只有 {v0['acc_pct']}%，低于 {MIN_START_ACC}%："
                                        "LoRA 没读对或者提示词拼错了，截图发到对话里")

# 2. 套一层新的 LoRA，准备训练
cfg = LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=0.0, target_modules="all-linear",
                 task_type="CAUSAL_LM")
md = get_peft_model(md, cfg)
md.print_trainable_parameters()
md.gradient_checkpointing_enable()
md.enable_input_require_grads()
opt = torch.optim.AdamW(trainable(md), lr=LR, weight_decay=0.0)

# 3. 训练题分批：同一批的题服务器台数相同
encode_rows(TRAIN_ROWS)
rng = np.random.default_rng(SEED)
batches = []
for b in batches_by_count(TRAIN_ROWS, PROMPTS_PER_STEP, rng):
    if len(b) == PROMPTS_PER_STEP:
        batches.append(b)
total = min(STEPS, len(batches))

# 4. 训练循环
log_f = open(os.path.join(OUT, "train_log.csv"), "w", newline="", encoding="utf-8")
log_w = None
STEP_LOG = []
t0 = time.time()
for step in range(total):
    st = grpo_step(md, opt, batches[step], step)
    st["elapsed_s"] = round(time.time() - t0, 1)
    STEP_LOG.append(st)
    if log_w is None:
        log_w = csv.DictWriter(log_f, fieldnames=list(st.keys()))
        log_w.writeheader()
    log_w.writerow(st)
    log_f.flush()
    if step == 19:
        eta = (time.time() - t0) / 20 * total
        print(f"前 20 步用了 {time.time() - t0:.0f} 秒，预计训练共约 {eta / 60:.0f} 分钟（估算）"
              + ("  ← 超过 45 分钟，停下来告诉我" if eta > 45 * 60 else ""))
    if (step + 1) % 50 == 0 or step == 0:
        print(f"第 {st['step']}/{total} 步  奖励 {st['reward_mean']}  采样命中 {st['sample_acc_pct']}%  "
              f"全相同组 {st['zero_std_frac']}  非法 {st['bad_server_pct'] + st['unparsed_pct']:.1f}%  "
              f"最优动作概率 {st['p_best_pct']}%  KL {st['kl']}  {st['elapsed_s'] / 60:.1f} 分钟")
    if (step + 1) % EVAL_EVERY == 0 or step + 1 == total:
        run_val(step + 1)
log_f.close()
SUMMARY["minutes_train"] = round((time.time() - t0) / 60, 1)
SUMMARY["val_log"] = VAL_LOG
with open(os.path.join(OUT, "val_log.json"), "w", encoding="utf-8") as f:
    json.dump(VAL_LOG, f, ensure_ascii=False, indent=1)
md.save_pretrained(os.path.join(OUT, "lora"))

# 5. 最终评测：四个集合全部，按部署规则作答，写 preds.csv（Mac 上 eval.open_loop 打分）
eval_rows = []
for split in EVAL_SPLITS:
    eval_rows += DATA[split]
preds = predict(md, eval_rows)
write_preds(os.path.join(OUT, "preds.csv"), eval_rows, preds)
with open(os.path.join(OUT, "config.json"), "w", encoding="utf-8") as f:
    json.dump({"name": "1.5b_hard + GRPO（fixed 奖励）", "model": MODEL_ID + " + hard_1.5b LoRA（合并）+ GRPO LoRA",
               "model_tag": "1.5B", "start": START_DIR, "steps": total, **SUMMARY["settings"]},
              f, ensure_ascii=False, indent=2)
SUMMARY["final_quick"] = {}
for split in EVAL_SPLITS:
    SUMMARY["final_quick"][split] = quick_metrics(DATA[split], preds)
    print("GRPO 后", split, SUMMARY["final_quick"][split])
with open(os.path.join(OUT, "summary.json"), "w", encoding="utf-8") as f:
    json.dump(SUMMARY, f, ensure_ascii=False, indent=1)
print("\n全部完成 ✓ 结果目录：", OUT)
print(sorted(os.listdir(OUT)))

In [ ]:
# 第 8 格：和起点对比（只打印，不判过不过）
#   1. 训练过程：前 100 步和后 100 步的平均奖励、奖励全相同的组的比例、最优动作的平均概率
#   2. 验证集：第 0 步 → 最后
#   3. 测试集：阶段 3 hard_1.5b 的 config.json 里记的准确率 → 这次 GRPO 后
def mean_of(logs, key):
    total_v = 0.0
    for x in logs:
        total_v += x[key]
    return total_v / len(logs)


k = min(100, len(STEP_LOG))
first = STEP_LOG[:k]
last = STEP_LOG[-k:]
print(f"训练（前 {k} 步 → 后 {k} 步）：")
for key, label in (("reward_mean", "平均奖励"), ("zero_std_frac", "奖励全相同的组的比例"),
                   ("p_best_pct", "最优动作的平均概率 %"), ("sample_acc_pct", "采样命中 %")):
    print(f"  {label}：{mean_of(first, key):.3f} → {mean_of(last, key):.3f}")
print(f"验证集准确率：第 0 步 {VAL_LOG[0]['acc_pct']}% → 最后 {VAL_LOG[-1]['acc_pct']}%")
start_cfg = os.path.join(START_DIR, "config.json")
if os.path.exists(start_cfg):
    with open(start_cfg, encoding="utf-8") as f:
        s3q = json.load(f).get("quick_metrics", {})
    print("测试集准确率（阶段 3 hard_1.5b → GRPO 后）：")
    for split in EVAL_SPLITS:
        if split in s3q:
            print(f"  {split}：{s3q[split]['acc_pct']}% → {SUMMARY['final_quick'][split]['acc_pct']}%")

## 跑完之后

1. 把第 7 格最后几行（"GRPO 后"那 4 行）和第 8 格的输出截图发到对话里。
2. 把整个 `hard_grpo_<UTC 时间>` 文件夹下载到 Mac 的 `runs/s4/` 里（`lora/` 子文件夹不用下载）。
3. 「代码执行程序 → 断开连接并删除运行时」。